In [ ]:
import os

In [ ]:
%pwd

In [ ]:
os.chdir("D:/Python AI ML DL/ML Flow e-t-e/E-to-E-ML-Project-using-ML-Flow/")

In [ ]:
%pwd

In [42]:
import os 

os.environ['MLFLOW_TRACKING_URI'] = 'https://dagshub.com/tirunap/E-to-E-ML-Project-using-ML-Flow.mlflow'
os.environ['MLFLOW_TRACKING_USERNAME'] = 'tirunap'
os.environ['MLFLOW_TRACKING_PASSWORD'] = '23Jan@1995'

In [57]:
from dataclasses import dataclass
from pathlib import Path
import os, sys
from src.mlProject import logger

@dataclass(frozen=True)
class TrainingPipelineConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_path: Path
    target_column: str
    mlflow_uri: str

In [58]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories, save_json

In [68]:
class ConfigurationManager:
    def __init__(self, 
                 config_filepath = CONFIG_FILE_PATH, 
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        
        create_directories([self.config.artifacts_root])
    
    def get_training_pipeline_config(self) -> TrainingPipelineConfig:
        training_pipeline_config = self.config.training_pipeline_config
        training_pipeline_config = TrainingPipelineConfig(
            root_dir=Path(training_pipeline_config.root_dir),
            test_data_path=Path(training_pipeline_config.test_data_path),
            model_path=Path(training_pipeline_config.model_path),
            all_params=self.params,
            metric_file_path=Path(training_pipeline_config.metric_file_path),
            target_column=self.config.target_column,
            mlflow_uri=self.config.mlflow_uri
        )
        create_directories([training_pipeline_config.root_dir])
        return training_pipeline_config

In [69]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib
from mlProject.utils.common import read_yaml, create_directories
from mlProject.entity.config_entity import TrainingPipelineConfig

In [70]:
class ModelEvaluation:
    def __init__(self, config: TrainingPipelineConfig):
        self.config = config


    def eval_metrics(self, actual, predicted):
        rmse = np.sqrt(mean_squared_error(actual, predicted))
        r2 = r2_score(actual, predicted)
        mae = mean_absolute_error(actual, predicted)
        return rmse, r2, mae
    
    def evaluate_model(self):
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop(self.config.target_column, axis=1)
        test_y = test_data[self.config.target_column]

        mlflow.set_tracking_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(self.config.mlflow_uri).scheme

        with mlflow.start_run():
            predicted = model.predict(test_x)

            rmse, r2, mae = self.eval_metrics(test_y, predicted)

            score = {"rmse": rmse, "r2": r2, "mae": mae}
            save_json(path=self.config.metric_file_path, data=score)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("r2", r2)
            mlflow.log_metric("mae", mae)

            mlflow.sklearn.log_model(model, "model")


            # Model registry does not work with file store
            if tracking_url_type_store != "file":

                #   Register the model
                #   There are other ways to use the Model Registry, which depends on the use case,
                #   please refer to the doc for more information:
                #   https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.sklearn.log_model(model, "model", registered_model_name="RandomForestRegressor")
            else:
                mlflow.sklearn.log_model(model, "model")    



In [73]:
try:
    config = ConfigurationManager()
    training_pipeline_config = config.get_training_pipeline_config()
    model_eval = ModelEvaluation(config=training_pipeline_config)
    model_eval.evaluate_model()

except Exception as e:
    raise e

[2026-04-20 21:28:26,046: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-20 21:28:26,081: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-20 21:28:26,088: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-04-20 21:28:26,092: INFO: common: created directory at: artifacts]
[2026-04-20 21:28:26,096: INFO: common: created directory at: artifacts\model_evaluation]
[2026-04-20 21:28:27,831: INFO: common: json file saved at: artifacts\model_evaluation\metric.txt]


2026/04/20 21:28:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/20 21:28:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/20 21:28:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/20 21:28:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

🏃 View run lyrical-lamb-729 at: https://dagshub.com/tirunap/E-to-E-ML-Project-using-ML-Flow.mlflow/#/experiments/0/runs/6504ac7c55c840e0a155783aae7b48bd
🧪 View experiment at: https://dagshub.com/tirunap/E-to-E-ML-Project-using-ML-Flow.mlflow/#/experiments/0
